# CoRe-TFM v4 — Reproducible Research Expansion

**Research status:** The historical experiment is not a real-data reconciliation win. This suite evaluates when (if ever) reconciliation improves on simple pools, view selection and calibration using *new real-model inferences*. The published method and original October 19 v2 cache protocol are preserved so the three verified Car/TabICLv2 caches can be reused. New outputs are kept separate from the frozen historical paper.

**Protocol limits:** Each `(dataset, seed, fold)` denotes a separately randomized held-out split, *not* a disjoint classical K-fold partition. `seed` and `fold` repeats are correlated within a dataset; the independent statistical unit is **dataset**. Fixing the val/test IDs across model and train-context sizes permits paired comparisons. The 205 validation labels alongside a 64-example context are an intentionally decoupled diagnostic budget, not a 64-total-label protocol.

**Scientific restrictions:** Never use test labels to choose temperatures, pooling weights, penalties or CoRe candidates. A pilot with one dataset cannot establish statistical significance. Model weight SHA-256 is not asserted unless exact checkpoint bytes are independently verified. Missing model access or CUDA is a blocked task, not a successful experiment.

References: [CoRe-TFM repository](https://github.com/bnssaanirudh/core-tfm); [TabICL API](https://tabicl.readthedocs.io/en/latest/api.html); [TabPFN official README](https://github.com/PriorLabs/TabPFN); [TabFM license](https://github.com/google-research/tabfm); [Scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html); Demšar (2006), [multiple-dataset statistical tests](https://jmlr.org/papers/v7/demsar06a.html).


In [ ]:
# 1 — SETUP. Edit these settings only. No paid inference API is required.
from pathlib import Path
import os, sys, subprocess, importlib.util, importlib.metadata as md, json, hashlib
IN_COLAB=('google.colab' in sys.modules)
MOUNT_DRIVE = IN_COLAB
OUTPUT_OVERRIDE = None  # e.g. '/content/drive/MyDrive/CoRe_TFM_Resumable_v2'
RECOVERY_ZIP = None   # e.g. '/content/CoReTFM_V2_PARTIAL_EVIDENCE (2).zip'; auto-search local uploads if None
INSTALL_MISSING_CPU_PACKAGES = True
AUTO_INSTALL_MODELS_ON_GPU = True
MODEL_PINS = {'tabiclv2':('tabicl','2.2.0'),'tabpfn3':('tabpfn','9.2.0')}
RUN_GPU_QUEUE = True
MAX_NEW_GPU_JOBS_PER_PASS = 1  # rerun queue cell on T4 to continue; use 2 only after stable pilot
RUN_REAL_ANALYSIS = True
RUN_FIGURES = True
RUN_ARCHIVE = True
DOWNLOAD_ZIP = False  # True downloads after archive cell; no need for extra download cell
VAL_BUDGETS = [16, 32, 52, 103, 205]  # changes selector budget only; never test ids
PRIMARY_CAP = 256
PRIMARY_VAL = 52
RUN_PROFILE = 'BALANCED'  # 'QUICK', 'BALANCED', 'FULL'
# QUICK is deliberately small; BALANCED prioritizes new datasets; FULL can take many Colab sessions.
if RUN_PROFILE == 'QUICK':
    DATASETS=['car','credit','wine'];SEEDS=[23];FOLDS=[0];TRAIN_CAPS=[128]
elif RUN_PROFILE == 'BALANCED':
    DATASETS=['car','credit','wine','phishing','nursery'];SEEDS=[23,42,71];FOLDS=[0,1];TRAIN_CAPS=[128,256,64]
elif RUN_PROFILE == 'FULL':
    DATASETS=['anneal','car','credit','customer','diamonds','marketing','mic','nursery','phishing','wine']
    SEEDS=[11,23,42,71,101];FOLDS=[0,1,2,3,4];TRAIN_CAPS=[64,128,256,512,1024]
else:raise ValueError('RUN_PROFILE must be QUICK, BALANCED or FULL')
MODELS=['tabiclv2','tabpfn3']
# Optional model: TabFM is not run here until model license, checkpoint and compatibility are independently established.
if MOUNT_DRIVE:
    from google.colab import drive
    try:drive.mount('/content/drive')
    except Exception as e:print('DRIVE_MOUNT_FAILED:',type(e).__name__,'— /content is ephemeral!')
if OUTPUT_OVERRIDE:
    os.environ['COREFM_OUT']=str(OUTPUT_OVERRIDE)
print('Colab:',IN_COLAB,'Profile:',RUN_PROFILE,'Planned datasets:',DATASETS)


In [ ]:
# 2 — Minimal CPU dependencies. GPU model installs happen only after a CUDA check.
required={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn','matplotlib':'matplotlib'}
missing=[pkg for module,pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING_CPU_PACKAGES:raise ImportError('Missing CPU dependencies '+str(missing))
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('CPU dependencies: OK')


In [ ]:
# 3 — Embedded v2 implementation. Standalone; preserves prior cache format.
import types
ENGINE_SOURCE = '"""CoRe-TFM resumable real-inference and evidence engine, protocol v2 (2026-10-19).\nStandalone implementation incorporating the paired-OpenML specification and four\nview construction from bnssaanirudh/core-tfm. New experiments must be analyzed\nseparately from the historical main-branch benchmark.\n"""\nfrom __future__ import annotations\nimport os, sys, json, hashlib, tempfile, time, traceback, gc, importlib.metadata as im\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom dataclasses import dataclass\nfrom typing import Any\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import wilcoxon\n\nPROTOCOL = "coretfm-v2-independent-fixed-test-2026-10-19"\nDATASETS = {\n    "anneal":(46906,"classes","steel"), "credit":(46918,"good_or_bad_customer","checking_status"),\n    "phishing":(46963,"WebsiteType","SFH"), "mic":(46980,"LET_IS","ASP_S_n"),\n    "customer":(46938,"bad_client_target","education"), "car":(40975,"class","safety"),\n    "marketing":(46940,"Response","Marital_Status"), "nursery":(26,"class","health"),\n    "diamonds":(42225,"cut","color"), "wine":([40691,40498],"class","color"),\n}\nCHECKPOINTS={"tabiclv2":"tabicl-classifier-v2-20260212.ckpt","tabpfn3":"ModelVersion.V3"}\n\n\ndef utcnow(): return datetime.now(timezone.utc).isoformat()\ndef sha_bytes(b): return hashlib.sha256(b).hexdigest()\ndef sha_obj(v): return sha_bytes(json.dumps(v,sort_keys=True,default=str).encode())\ndef _atomic_bytes(path:Path, data:bytes):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent,prefix=\'.tmp_\',delete=False) as f:\n        f.write(data); tmp=Path(f.name)\n    os.replace(tmp,path)\ndef put_json(path,payload): _atomic_bytes(Path(path),json.dumps(payload,indent=2,sort_keys=True,default=str).encode())\ndef get_json(path): return json.loads(Path(path).read_text())\ndef out_root():\n    custom=os.environ.get(\'COREFM_OUT\')\n    if custom: return Path(custom).expanduser().resolve()\n    mounted=Path(\'/content/drive/MyDrive\')\n    if mounted.exists() and mounted.is_dir(): return mounted/\'CoRe_TFM_Resumable_v2\'\n    return Path.cwd()/\'CoRe_TFM_Resumable_v2_results\'\ndef init_dirs(root=None):\n    root=Path(root) if root is not None else out_root()\n    for k in [\'datasets\',\'splits\',\'real_caches\',\'smoke_caches\',\'reports\',\'runtime\',\'failures\']:(root/k).mkdir(parents=True,exist_ok=True)\n    return root\n\ndef cuda_status():\n    try:\n        import torch\n        ok=bool(torch.cuda.is_available())\n        return {\'cuda\':ok,\'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0) if ok else None}\n    except Exception as e: return {\'cuda\':False,\'reason\':repr(e)}\n\ndef version(package):\n    try: return im.version(package)\n    except im.PackageNotFoundError: return \'not_installed\'\n\n\ndef load_pair_dataset(name,root=None,download=True):\n    """Pinned two-outcome OpenML pair; dataset bytes cached on Drive as pickle.\n    Missing values and derived features follow the repository\'s openml.py policy.\n    No label from validation/test ever enters model input other than hypothetical\n    conditioning values enumerated over every possible category.\n    """\n    if name not in DATASETS:raise KeyError(name)\n    root=init_dirs(root);loc=root/\'datasets\'/f\'{name}.pkl\'\n    if loc.exists():\n        data=pd.read_pickle(loc)\n        return data[\'X\'],data[\'a\'],data[\'b\'],data[\'source\']\n    if not download: raise FileNotFoundError(loc)\n    from sklearn.datasets import fetch_openml\n    ids,ta,tb=DATASETS[name]\n    if name==\'wine\':\n        cols=[\'fixed_acidity\',\'volatile_acidity\',\'citric_acid\',\'residual_sugar\',\'chlorides\',\'free_sulfur_dioxide\',\'total_sulfur_dioxide\',\'density\',\'pH\',\'sulphates\',\'alcohol\']\n        parts=[]\n        for oid,c in ((40691,\'red\'),(40498,\'white\')):\n            ds=fetch_openml(data_id=oid,as_frame=True,parser=\'auto\')\n            x=ds.data.copy();x.columns=cols;x[\'__a__\']=pd.Series(ds.target).to_numpy();x[\'__b__\']=c;parts.append(x)\n        frame=pd.concat(parts,ignore_index=True)\n        source={\'openml_ids\':[40691,40498],\'target_a\':ta,\'target_b\':tb}\n    else:\n        ds=fetch_openml(data_id=ids,as_frame=True,parser=\'auto\')\n        x=ds.data.copy()\n        if tb not in x:raise ValueError(f\'{name}: expected target B {tb} absent; columns={list(x.columns)[:15]}\')\n        x[\'__a__\']=np.asarray(ds.target);x[\'__b__\']=x.pop(tb)\n        if name==\'mic\':\n            features=x.drop(columns=[\'__a__\',\'__b__\'])\n            x=x.loc[:, list(features.columns[features.isna().mean()<=.02])+[\'__a__\',\'__b__\']]\n        if name==\'marketing\':\n            for col in x.columns:\n                if \'date\' in str(col).lower() or \'enroll\' in str(col).lower():\n                    parsed=pd.to_datetime(x[col],errors=\'coerce\',dayfirst=True)\n                    if parsed.notna().mean()>.95:\n                        x[col]=(parsed-parsed.min()).dt.days.astype(float);break\n        frame=x\n        source={\'openml_id\':ids,\'target_a\':ta,\'target_b\':tb}\n    frame=frame.dropna(axis=0).reset_index(drop=True)\n    a=frame.pop(\'__a__\').astype(\'category\'); b=frame.pop(\'__b__\').astype(\'category\')\n    X=frame.copy()\n    # NOTE: no leakage - targets are removed from predictors.\n    if X.columns.duplicated().any():raise ValueError(\'duplicate feature names\')\n    data={\'X\':X,\'a\':a,\'b\':b,\'source\':source}\n    pd.to_pickle(data,loc)\n    return X,a,b,source\n\n\ndef coverage_order(indices, a, b, rng):\n    """Deterministically frontload both marginal class coverage for nested caps."""\n    seq=np.array(indices,dtype=int).copy();rng.shuffle(seq)\n    all_a=set(a.iloc[seq].astype(str));all_b=set(b.iloc[seq].astype(str))\n    seen_a=set();seen_b=set();picks=[];unused=list(seq)\n    while (seen_a!=all_a or seen_b!=all_b) and unused:\n        gains=[(int(str(a.iloc[i]) not in seen_a)+int(str(b.iloc[i]) not in seen_b)) for i in unused]\n        pos=int(np.argmax(gains)); i=unused.pop(pos)\n        picks.append(i);seen_a.add(str(a.iloc[i]));seen_b.add(str(b.iloc[i]))\n    return np.array(picks+unused,dtype=int),len(picks)\n\n\ndef make_split(name,seed=23,fold=0,root=None,test_n=128,val_n=205):\n    from sklearn.model_selection import train_test_split\n    root=init_dirs(root);X,a,b,source=load_pair_dataset(name,root)\n    path=root/\'splits\'/f\'{name}__seed{seed}__fold{fold}.json\'\n    if path.exists():\n        p=get_json(path)\n        if (p.get(\'protocol\')!=PROTOCOL or p[\'n_total\']!=len(X) or p.get(\'requested_test_n\')!=test_n or p.get(\'requested_val_n\')!=val_n):\n            raise ValueError(\'split incompatible with protocol, dataset size or test/validation reservation; refusing overwrite\')\n        return p\n    if len(X)<test_n+val_n+65:raise ValueError(f\'{name}: insufficient rows for fixed test/validation split\')\n    idx=np.arange(len(X));r=seed*1009+fold*100003\n    # Deterministic random outer split; no stratification promise for very rare joint classes.\n    pool,test=train_test_split(idx,test_size=test_n,random_state=r,shuffle=True)\n    train_pool,val=train_test_split(pool,test_size=val_n,random_state=r+17,shuffle=True)\n    ordered,coverage=coverage_order(train_pool,a,b,np.random.default_rng(r+29))\n    vordered=np.array(val,dtype=int);np.random.default_rng(r+31).shuffle(vordered)\n    split={\'protocol\':PROTOCOL,\'dataset\':name,\'seed\':seed,\'fold\':fold,\'n_total\':len(X),\'source\':source,\n           \'train_pool_ids\':ordered.tolist(),\'val_ids\':vordered.tolist(),\'test_ids\':np.array(test,dtype=int).tolist(),\n           \'coverage_frontload_rows\':coverage,\'requested_test_n\':test_n,\'requested_val_n\':val_n,\'classes_a\':list(map(str,a.cat.categories)),\n           \'classes_b\':list(map(str,b.cat.categories))}\n    assert_no_overlap(split)\n    split[\'split_sha256\']=sha_obj({k:v for k,v in split.items() if k!=\'split_sha256\'})\n    put_json(path,split);return split\n\ndef assert_no_overlap(s):\n    i=[set(map(int,s[k])) for k in [\'train_pool_ids\',\'val_ids\',\'test_ids\']]\n    if any(len(set(s[k]))!=len(s[k]) for k in [\'train_pool_ids\',\'val_ids\',\'test_ids\']):raise AssertionError(\'duplicate IDs within split\')\n    if i[0]&i[1] or i[0]&i[2] or i[1]&i[2]:raise AssertionError(\'train/val/test overlap\')\n\ndef _condition_col(X,name,values,value):\n    v=X.copy(); v[name]=pd.Categorical([str(value)]*len(v),categories=list(map(str,values)))\n    return v\n\ndef _aligned_proba(est,X,classes):\n    pred=np.asarray(est.predict_proba(X),dtype=float)\n    order={str(c):i for i,c in enumerate(np.asarray(est.classes_).tolist())}\n    if set(order)!=set(map(str,classes)):raise RuntimeError(\'model classes differ from the frozen global label domain\')\n    pred=pred[:,[order[str(c)] for c in classes]]\n    if pred.shape!=(len(X),len(classes)) or not np.isfinite(pred).all():raise ValueError(\'invalid model predict_proba\')\n    if np.min(pred)<-1e-9 or np.max(pred)>1+1e-9:raise ValueError(\'out-of-range probability\')\n    pred=np.maximum(pred,0)\n    if (pred.sum(axis=1)<=0).any():raise ValueError(\'zero probability mass\')\n    return pred/pred.sum(axis=1,keepdims=True)\n\ndef make_factory(model,device=\'cuda\',seed=23):\n    if model==\'tabiclv2\':\n        from tabicl import TabICLClassifier\n        def f():return TabICLClassifier(n_estimators=2,kv_cache=False,batch_size=2,checkpoint_version=CHECKPOINTS[\'tabiclv2\'],device=device,random_state=seed)\n        return f\n    if model==\'tabpfn3\':\n        from tabpfn import TabPFNClassifier\n        from tabpfn.constants import ModelVersion\n        if not hasattr(ModelVersion,\'V3\'):raise RuntimeError(\'TabPFN V3 not supported by installed package\')\n        def f():\n            obj=TabPFNClassifier.create_default_for_version(ModelVersion.V3)\n            p=obj.get_params();changes={k:v for k,v in {\'n_estimators\':2,\'device\':device,\'random_state\':seed}.items() if k in p}\n            if changes:obj.set_params(**changes)\n            return obj\n        return f\n    if model==\'smoke_logreg\':\n        from sklearn.compose import ColumnTransformer\n        from sklearn.preprocessing import OneHotEncoder,StandardScaler\n        from sklearn.impute import SimpleImputer\n        from sklearn.pipeline import make_pipeline\n        from sklearn.linear_model import LogisticRegression\n        def f():\n            # Fully self-contained smoke model; never publish as a TFM run.\n            return make_pipeline(ColumnTransformer([(\'category\',make_pipeline(SimpleImputer(strategy=\'most_frequent\'),OneHotEncoder(handle_unknown=\'ignore\')),lambda df:list(df.select_dtypes(exclude=\'number\').columns)),(\'numeric\',make_pipeline(SimpleImputer(strategy=\'median\'),StandardScaler()),lambda df:list(df.select_dtypes(include=\'number\').columns))],remainder=\'drop\'),LogisticRegression(max_iter=300))\n        return f\n    raise ValueError(f\'Unknown model: {model}\')\n\ndef four_views(factory,Xtrain,atrain,btrain,Xval,Xtest,classes_a,classes_b):\n    """Fit four estimators with distinct views; return all arrays and fit/predict timings.\n    Only training labels enter conditional training features. For prediction each\n    possible conditioning value is enumerated; test ground truth is never used.\n    """\n    nva=len(Xval);nte=len(Xtest);ka=len(classes_a);kb=len(classes_b)\n    trainA=pd.Series(pd.Categorical(atrain.astype(str),categories=classes_a))\n    trainB=pd.Series(pd.Categorical(btrain.astype(str),categories=classes_b))\n    Xa=Xtrain.reset_index(drop=True).copy();V=Xval.reset_index(drop=True);T=Xtest.reset_index(drop=True)\n    if any(c in Xa for c in [\'__cond_A__\',\'__cond_B__\']):raise ValueError(\'reserved feature collision\')\n    views={};times={};\n    def view(key,y,extra_col=None,extra_vals=None,condition_on=None):\n        if extra_col is None:\n            train=Xa; xvs=[V];xts=[T]\n        else:\n            train=_condition_col(Xa,extra_col,extra_vals,np.nan) # overwritten below\n            train[extra_col]=pd.Categorical(np.asarray(condition_on).astype(str),categories=list(map(str,extra_vals)))\n            xvs=[_condition_col(V,extra_col,extra_vals,c) for c in extra_vals]\n            xts=[_condition_col(T,extra_col,extra_vals,c) for c in extra_vals]\n        start=time.perf_counter();est=factory();est.fit(train,y);fit_secs=time.perf_counter()-start\n        pred_start=time.perf_counter()\n        expected=classes_a if key in (\'a\',\'a_given_b\') else classes_b\n        pv=np.stack([_aligned_proba(est,x,expected) for x in xvs],axis=1)\n        pt=np.stack([_aligned_proba(est,x,expected) for x in xts],axis=1)\n        inference_secs=time.perf_counter()-pred_start\n        if pv.shape[1]==1:pv=pv[:,0,:];pt=pt[:,0,:]\n        views[key+\'_val\']=pv;views[key+\'_test\']=pt\n        times[key]={\'fit_with_load_sec\':fit_secs,\'predict_sec\':inference_secs,\'n_predict_calls\':len(xvs)+len(xts)}\n        del est;gc.collect()\n        try:\n            import torch\n            if torch.cuda.is_available():torch.cuda.empty_cache()\n        except ImportError:pass\n    view(\'a\',trainA)\n    view(\'b\',trainB)\n    view(\'a_given_b\',trainA,extra_col=\'__cond_B__\',extra_vals=classes_b,condition_on=trainB)\n    view(\'b_given_a\',trainB,extra_col=\'__cond_A__\',extra_vals=classes_a,condition_on=trainA)\n    assert views[\'a_given_b_val\'].shape==(nva,kb,ka)\n    assert views[\'b_given_a_val\'].shape==(nva,ka,kb)\n    assert views[\'a_given_b_test\'].shape==(nte,kb,ka)\n    return views,times\n\ndef _validate_probability_view(arr,axis=-1):\n    arr=np.asarray(arr,dtype=float)\n    if not np.all(np.isfinite(arr)) or (arr<-1e-8).any():raise ValueError(\'invalid probabilities\')\n    if not np.allclose(arr.sum(axis=axis),1,atol=1e-4):raise ValueError(\'probability rows not normalized\')\n\ndef run_case(dataset,model,seed=23,fold=0,train_cap=64,root=None,test_n=128,val_n=205,allow_cpu=False):\n    root=init_dirs(root)\n    if model!=\'smoke_logreg\' and not cuda_status()[\'cuda\'] and not allow_cpu:\n        return {\'status\':\'SKIPPED_NO_GPU\',\'dataset\':dataset,\'model\':model,\'explanation\':\'Run in a CUDA Colab session, or explicitly allow CPU (slow).\'}\n    split=make_split(dataset,seed,fold,root,test_n=test_n,val_n=val_n)\n    assert_no_overlap(split)\n    ids=np.asarray(split[\'train_pool_ids\'],dtype=int)\n    if train_cap>len(ids):return {\'status\':\'SKIPPED_INSUFFICIENT_TRAIN_POOL\',\'train_cap\':train_cap,\'train_pool_size\':len(ids)}\n    if train_cap<split[\'coverage_frontload_rows\']:\n        return {\'status\':\'SKIPPED_CLASS_COVERAGE\',\'train_cap\':train_cap,\'minimum_required\':split[\'coverage_frontload_rows\']}\n    filekey=f\'{dataset}__{model}__seed{seed}__fold{fold}__train{train_cap}\'\n    cdir=root/(\'smoke_caches\' if model==\'smoke_logreg\' else \'real_caches\')\n    path=cdir/(filekey+\'.npz\')\n    if path.exists():\n        try:\n            _,meta=read_cache(path)\n            if meta[\'split_sha256\']==split[\'split_sha256\'] and meta[\'protocol\']==PROTOCOL and meta[\'train_cap\']==train_cap and meta[\'model\']==model:\n                pkg={\'tabiclv2\':\'tabicl\',\'tabpfn3\':\'tabpfn\',\'smoke_logreg\':\'scikit-learn\'}[model]\n                current=version(pkg)\n                if current not in (\'not_installed\',meta[\'package_version\']):\n                    return {\'status\':\'CACHED_VERSION_CONFLICT\',\'path\':str(path),\'cached_package_version\':meta[\'package_version\'],\'current_package_version\':current}\n                return {\'status\':\'CACHED_VERIFIED\',\'path\':str(path)}\n        except Exception as ex:raise RuntimeError(f\'Existing cache is corrupt or incompatible; DO NOT overwrite: {path}: {ex}\')\n    X,a,b,source=load_pair_dataset(dataset,root)\n    tr=ids[:train_cap];va=np.array(split[\'val_ids\'],dtype=int);te=np.array(split[\'test_ids\'],dtype=int)\n    ca=split[\'classes_a\'];cb=split[\'classes_b\']\n    if set(map(str,a.iloc[tr]))!=set(ca) or set(map(str,b.iloc[tr]))!=set(cb):\n        return {\'status\':\'SKIPPED_CLASS_COVERAGE\',\'dataset\':dataset,\'train_cap\':train_cap}\n    first=time.perf_counter()\n    arrays,times=four_views(make_factory(model,device=\'cuda\' if cuda_status()[\'cuda\'] else \'cpu\',seed=seed),X.iloc[tr],a.iloc[tr].reset_index(drop=True),b.iloc[tr].reset_index(drop=True),X.iloc[va],X.iloc[te],ca,cb)\n    for k,v in arrays.items():_validate_probability_view(v)\n    lookup_a={c:i for i,c in enumerate(ca)};lookup_b={c:i for i,c in enumerate(cb)}\n    for role,ix in [(\'val\',va),(\'test\',te)]:\n        arrays[f\'ya_{role}\']=np.array([lookup_a[str(v)] for v in a.iloc[ix]],dtype=np.int32)\n        arrays[f\'yb_{role}\']=np.array([lookup_b[str(v)] for v in b.iloc[ix]],dtype=np.int32)\n    for role,ix in [(\'train\',tr),(\'val\',va),(\'test\',te)]:arrays[role+\'_ids\']=np.array(ix,dtype=np.int32)\n    meta={\'protocol\':PROTOCOL,\'dataset\':dataset,\'model\':model,\'artifact_kind\':\'smoke_only\' if model==\'smoke_logreg\' else \'real_tfm\',\n          \'seed\':seed,\'fold\':fold,\'train_cap\':train_cap,\'n_train\':len(tr),\'n_val\':len(va),\'n_test\':len(te),\n          \'split_sha256\':split[\'split_sha256\'],\'train_ids_sha\':sha_obj(tr.tolist()),\'val_ids_sha\':sha_obj(va.tolist()),\n          \'test_ids_sha\':sha_obj(te.tolist()),\'classes_a\':ca,\'classes_b\':cb,\n          \'package_version\':version({\'tabiclv2\':\'tabicl\',\'tabpfn3\':\'tabpfn\',\'smoke_logreg\':\'scikit-learn\'}[model]),\n          \'checkpoint_identifier\':CHECKPOINTS.get(model,\'not_a_TFM\'),\n          \'checkpoint_sha256\':\'NOT_VERIFIED\',\'checkpoint_revision_confirmed\':False,\n          \'device\':cuda_status(),\'timing\':times,\'elapsed_s\':time.perf_counter()-first,\'created_utc\':utcnow(),\n          \'source\':source,\'n_probabilities\':sum(np.size(x) for x in arrays.values()),\n          \'evidence_caveat\':\'Checkpoint filename recorded; actual weight-byte SHA256 not obtained.\'}\n    with tempfile.NamedTemporaryFile(dir=cdir,prefix=\'.npz_tmp_\',suffix=\'.npz\',delete=False) as f:tmp=Path(f.name)\n    try:\n        np.savez_compressed(tmp,**arrays)\n        with np.load(tmp,allow_pickle=False) as test:\n            if set(test.files)!=set(arrays):raise AssertionError(\'NPZ key mismatch\')\n        meta[\'npz_sha256\']=sha_bytes(tmp.read_bytes())\n        os.replace(tmp,path)\n        put_json(path.with_suffix(\'.json\'),meta)\n    finally:\n        if tmp.exists():tmp.unlink()\n    return {\'status\':\'REAL_MODEL_CACHE_WRITTEN\' if model!=\'smoke_logreg\' else \'SMOKE_ONLY_CACHE_WRITTEN\',\'path\':str(path),\'n_train\':len(tr),\'n_val\':len(va),\'n_test\':len(te),\'elapsed_sec\':meta[\'elapsed_s\']}\n\ndef read_cache(path):\n    path=Path(path);side=path.with_suffix(\'.json\')\n    if not side.exists():raise FileNotFoundError(f\'Cache sidecar missing: {side}\')\n    meta=get_json(side)\n    if meta.get(\'npz_sha256\')!=sha_bytes(path.read_bytes()):raise ValueError(f\'Cache checksum mismatch: {path}\')\n    if meta.get(\'protocol\')!=PROTOCOL:raise ValueError(\'incompatible protocol\')\n    with np.load(path,allow_pickle=False) as npz: data={k:npz[k] for k in npz.files}\n    expected=[\'a_val\',\'b_val\',\'a_given_b_val\',\'b_given_a_val\',\'a_test\',\'b_test\',\'a_given_b_test\',\'b_given_a_test\',\'ya_val\',\'yb_val\',\'ya_test\',\'yb_test\',\'train_ids\',\'val_ids\',\'test_ids\']\n    if set(expected)-set(data):raise ValueError(\'missing probability arrays/labels/ids\')\n    if len(data[\'a_val\'])!=meta[\'n_val\'] or len(data[\'a_test\'])!=meta[\'n_test\']:raise ValueError(\'cache length mismatch\')\n    for k in [\'a_val\',\'b_val\',\'a_given_b_val\',\'b_given_a_val\',\'a_test\',\'b_test\',\'a_given_b_test\',\'b_given_a_test\']:_validate_probability_view(data[k])\n    if len(set(data[\'train_ids\']) & set(data[\'val_ids\'])) or len(set(data[\'train_ids\']) & set(data[\'test_ids\'])) or len(set(data[\'test_ids\']) & set(data[\'val_ids\'])):raise ValueError(\'cache train/test leakage\')\n    if sha_obj(data[\'test_ids\'].tolist())!=meta[\'test_ids_sha\']:raise ValueError(\'test ids hash invalid\')\n    if sha_obj(data[\'val_ids\'].tolist())!=meta[\'val_ids_sha\']:raise ValueError(\'validation ids hash invalid\')\n    return data,meta\n\ndef normalize(x):\n    x=np.maximum(np.asarray(x,dtype=float),1e-12)\n    return x/x.sum(axis=(1,2),keepdims=True)\n\ndef implied(data,suffix):\n    pa=data[\'a_\'+suffix];pb=data[\'b_\'+suffix];pab=data[\'a_given_b_\'+suffix];pba=data[\'b_given_a_\'+suffix]\n    j1=np.transpose(pab,(0,2,1))*pb[:,None,:]\n    j2=pa[:,:,None]*pba\n    return normalize(j1),normalize(j2)\n\ndef hard_project(j,pa,pb,max_iter=150,tol=1e-9):\n    q=normalize(j);pa=np.maximum(pa,1e-12);pa=pa/pa.sum(axis=1,keepdims=True);pb=np.maximum(pb,1e-12);pb=pb/pb.sum(axis=1,keepdims=True)\n    for _ in range(max_iter):\n        q=q*(pa[:,:,None]/np.maximum(q.sum(axis=2,keepdims=True),1e-15))\n        q=q*(pb[:,None,:]/np.maximum(q.sum(axis=1,keepdims=True),1e-15))\n        if max(np.max(np.abs(q.sum(axis=2)-pa)),np.max(np.abs(q.sum(axis=1)-pb)))<tol:break\n    return normalize(q)\n\ndef candidates(data,suffix):\n    j1,j2=implied(data,suffix)\n    arith=normalize(.5*j1+.5*j2)\n    geo=normalize(np.sqrt(np.maximum(j1,1e-12)*np.maximum(j2,1e-12)))\n    hard=hard_project(arith,data[\'a_\'+suffix],data[\'b_\'+suffix])\n    return {\'arithmetic\':arith,\'raw_BA\':j1,\'raw_AB\':j2,\'geometric\':geo,\'hard\':hard,\n            \'soft_25\':normalize(.75*arith+.25*hard),\'soft_50\':normalize(.5*arith+.5*hard),\n            \'soft_75\':normalize(.25*arith+.75*hard)}\n\ndef per_sample_metrics(j,ya,yb):\n    flat=j.reshape(len(j),-1);ka,kb=j.shape[1:]\n    y=np.asarray(ya)*kb+np.asarray(yb)\n    nll=-np.log(np.maximum(flat[np.arange(len(y)),y],1e-12))\n    brier=(flat*flat).sum(axis=1)+1-2*flat[np.arange(len(y)),y]\n    return nll,brier\n\ndef select_method(cand_val,ya,yb,n_val,policy=\'full\',beta=.5):\n    m=min(int(n_val),len(ya))\n    if m<4:raise ValueError(\'too few validation labels\')\n    keys={\'pool\':[\'arithmetic\',\'geometric\'],\'full\':list(cand_val),\'regularized\':list(cand_val)}[policy]\n    losses={k:float(per_sample_metrics(cand_val[k][:m],ya[:m],yb[:m])[0].mean()) for k in keys}\n    best=min(losses,key=losses.get)\n    if policy==\'regularized\':\n        # Predeclared finite-validation guard; no test tuning and no confidence claim.\n        pen=beta*np.sqrt(2*np.log(max(len(keys),2))/m)\n        if losses[\'arithmetic\']-losses[best]<=pen:best=\'arithmetic\'\n    return best,losses\n\ndef score_cache(path,budgets=(16,32,52,103,205)):\n    d,meta=read_cache(path)\n    if meta.get(\'artifact_kind\')!=\'real_tfm\':raise ValueError(\'Smoke cache is not scientific real-TFM evidence\')\n    V=candidates(d,\'val\');T=candidates(d,\'test\')\n    yaV=d[\'ya_val\'];ybV=d[\'yb_val\'];yaT=d[\'ya_test\'];ybT=d[\'yb_test\']\n    j1,j2=implied(d,\'test\')\n    tv=float((.5*np.abs(j1-j2).sum(axis=(1,2))).mean())\n    rows=[]\n    for v in budgets:\n        if v>len(yaV):continue\n        for policy in [\'pool\',\'full\',\'regularized\']:\n            chosen,vloss=select_method(V,yaV,ybV,v,policy=policy)\n            nll,brier=per_sample_metrics(T[chosen],yaT,ybT)\n            pool_nll,pool_brier=per_sample_metrics(T[\'arithmetic\'],yaT,ybT)\n            rows.append({\'dataset\':meta[\'dataset\'],\'model\':meta[\'model\'],\'seed\':meta[\'seed\'],\'fold\':meta[\'fold\'],\n                         \'train_cap\':meta[\'train_cap\'],\'val_budget\':int(v),\'policy\':policy,\'selected\':chosen,\n                         \'joint_nll\':float(nll.mean()),\'joint_brier\':float(brier.mean()),\n                         \'pool_nll\':float(pool_nll.mean()),\'pool_brier\':float(pool_brier.mean()),\n                         \'delta_nll\':float((nll-pool_nll).mean()),\'delta_brier\':float((brier-pool_brier).mean()),\n                         \'factor_tv\':tv,\'n_val\':len(yaV),\'n_test\':len(yaT),\'test_ids_sha\':meta[\'test_ids_sha\'],\n                         \'split_sha256\':meta[\'split_sha256\'],\'cache_file\':str(path),\n                         \'checkpoint_revision_confirmed\':meta[\'checkpoint_revision_confirmed\'],\n                         \'checkpoint_sha256\':meta[\'checkpoint_sha256\']})\n    # Additional direct marginal and conditional diagnostics (proper scores).\n    marginal=[]\n    for label,direct,cond in [(\'A\',d[\'a_test\'],d[\'a_given_b_test\'][np.arange(len(yaT)),ybT,:]),\n                              (\'B\',d[\'b_test\'],d[\'b_given_a_test\'][np.arange(len(yaT)),yaT,:])]:\n        y=yaT if label==\'A\' else ybT\n        for kind,p in [(\'direct\',direct),(\'conditional\',cond)]:\n            nll=-np.log(np.maximum(p[np.arange(len(y)),y],1e-12))\n            brier=(p*p).sum(axis=1)+1-2*p[np.arange(len(y)),y]\n            marginal.append({\'dataset\':meta[\'dataset\'],\'model\':meta[\'model\'],\'seed\':meta[\'seed\'],\'fold\':meta[\'fold\'],\n                             \'train_cap\':meta[\'train_cap\'],\'target\':label,\'view\':kind,\'nll\':float(nll.mean()),\'brier\':float(brier.mean())})\n    return rows,marginal\n\ndef analyze_real(root=None,budgets=(16,32,52,103,205)):\n    root=init_dirs(root); paths=sorted((root/\'real_caches\').glob(\'*.npz\'))\n    if not paths:\n        result={\'status\':\'BLOCKED_NO_REAL_CACHE\',\'real_cache_count\':0,\'explanation\':\'Run NB36 or NB37 in GPU Colab first.\'}\n        put_json(root/\'reports\'/\'analysis_status.json\',result);return result\n    rows=[];marginal=[];bad=[]\n    for p in paths:\n        try:\n            r,m=score_cache(p,budgets);rows.extend(r);marginal.extend(m)\n        except Exception as e:bad.append({\'path\':str(p),\'error\':str(e)})\n    if not rows:\n        status={\'status\':\'BLOCKED_NO_VALID_CACHE\',\'errors\':bad}\n        put_json(root/\'reports\'/\'analysis_status.json\',status);return status\n    df=pd.DataFrame(rows);m=pd.DataFrame(marginal)\n    # For a given outer split/seed/fold the test IDs MUST stay fixed across caps and models.\n    same=df.groupby([\'dataset\',\'seed\',\'fold\'])[\'test_ids_sha\'].nunique()\n    if (same>1).any():raise AssertionError(f\'fixed test violation: {same[same>1].to_dict()}\')\n    df.to_csv(root/\'reports\'/\'fixed_test_factorial.csv\',index=False)\n    m.to_csv(root/\'reports\'/\'marginal_conditional_scores.csv\',index=False)\n    # Dataset-blocked inference: collapse folds and seeds/models within dataset.\n    blocks=df.groupby([\'dataset\',\'train_cap\',\'val_budget\',\'policy\'],as_index=False).agg(delta_nll=(\'delta_nll\',\'mean\'),delta_brier=(\'delta_brier\',\'mean\'))\n    blocks.to_csv(root/\'reports\'/\'dataset_blocks.csv\',index=False)\n    inf=[]\n    rng=np.random.default_rng(20261019)\n    for key,g in blocks.groupby([\'train_cap\',\'val_budget\',\'policy\']):\n        deltas=g.delta_nll.to_numpy();n=len(deltas)\n        ci=(np.nan,np.nan);p=np.nan\n        if n>=2:\n            bootstrap=rng.choice(deltas,size=(3000,n),replace=True).mean(axis=1)\n            ci=np.quantile(bootstrap,[.025,.975]);\n        if n>=5 and np.any(np.abs(deltas)>1e-12):\n            try:p=float(wilcoxon(deltas,alternative=\'two-sided\').pvalue)\n            except ValueError:pass\n        inf.append(dict(zip([\'train_cap\',\'val_budget\',\'policy\'],key))|{\'datasets\':n,\'mean_delta_nll\':float(deltas.mean()),\'ci95_low\':float(ci[0]),\'ci95_high\':float(ci[1]),\'wilcoxon_p_exploratory\':p,\n                          \'inferential_note\':\'Exploratory unless independent datasets are sufficient; hyperparameters fixed before test.\'})\n    pd.DataFrame(inf).to_csv(root/\'reports\'/\'dataset_blocked_uncertainty.csv\',index=False)\n    result={\'status\':\'ANALYZED_REAL_CACHES\',\'real_cache_count\':len(paths),\'valid_real_cache_count\':len(paths)-len(bad),\'invalid_cache_count\':len(bad),\n            \'method_rows\':len(df),\'datasets\':int(df.dataset.nunique()),\'models\':sorted(df.model.unique().tolist()),\'errors\':bad,\n            \'reports_dir\':str(root/\'reports\'),\'caveat\':\'Exploratory pilot and model weights not authenticated unless checkpoint hashes are separately verified.\'}\n    put_json(root/\'reports\'/\'analysis_status.json\',result);return result\n\ndef summarize_runtime(root=None):\n    root=init_dirs(root);rows=[]\n    for path in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            _,meta=read_cache(path)\n            t=meta[\'timing\']; direct=sum(t[k][\'fit_with_load_sec\']+t[k][\'predict_sec\'] for k in [\'a\',\'b\'])\n            four=direct+sum(t[k][\'fit_with_load_sec\']+t[k][\'predict_sec\'] for k in [\'a_given_b\',\'b_given_a\'])\n            rows.append({\'dataset\':meta[\'dataset\'],\'model\':meta[\'model\'],\'seed\':meta[\'seed\'],\'fold\':meta[\'fold\'],\n                         \'train_cap\':meta[\'train_cap\'],\'direct_two_view_wall_s\':direct,\n                         \'four_view_wall_s\':four,\'four_to_two_ratio\':four/direct if direct else np.nan,\n                         \'n_test\':meta[\'n_test\'],\'timing_type\':\'sequential end-to-end estimator fit+predict; includes possible checkpoint loading/download\',\n                         \'gpu\':meta[\'device\'].get(\'gpu\')})\n        except Exception as e: rows.append({\'cache\':str(path),\'error\':str(e)})\n    if rows:pd.DataFrame(rows).to_csv(root/\'runtime\'/\'cache_runtime.csv\',index=False)\n    return {\'status\':\'RUNTIME_SUMMARY\' if rows else \'NOT_RUN_NO_REAL_CACHES\',\'rows\':len(rows),\'path\':str(root/\'runtime\'/\'cache_runtime.csv\')}\n\ndef controlled_stress(root=None,repeats=8,start_repeat=0,validation_sizes=(8,16,32,64,128,256)):\n    """Checkpointed CPU known-truth study, a small number of repeats per call.\n    A new call appends only unique (regime,repeat,n_validation) cases. Deterministic\n    per-repeat seeds make results independent of execution interruption/batching.\n    """\n    root=init_dirs(root)\n    n=500; true=np.repeat(np.array([[[.65,.10],[.10,.15]]]),n,axis=0)\n    rows=[]\n    regimes=[\'marginal_reliable\',\'marginal_noisy\',\'shifted_validation\']\n    for regime_idx,regime in enumerate(regimes):\n        for r in range(start_repeat,start_repeat+repeats):\n            rng=np.random.default_rng(20261019+10001*r+regime_idx)\n            noise=rng.normal(0,.13,(n,2,2));v1=normalize(np.maximum(true+noise,1e-5));v2=normalize(np.maximum(true-noise,1e-5))\n            pooled=normalize(.5*v1+.5*v2)\n            if regime==\'marginal_reliable\':\n                pa=true.sum(axis=2);pb=true.sum(axis=1)\n            else:\n                fake=normalize(np.maximum(true+rng.normal(0,.16,(n,2,2)),1e-5))\n                pa=fake.sum(axis=2);pb=fake.sum(axis=1)\n            hard=hard_project(pooled,pa,pb)\n            for vb in validation_sizes:\n                rng_idx=np.random.default_rng(300001+1001*r+vb+regime_idx*100000)\n                yval=rng_idx.choice(4,size=vb,p=true[0].reshape(-1))\n                ytest=rng_idx.choice(4,size=200,p=true[0].reshape(-1))\n                va_pool=pooled[:vb];va_hard=hard[:vb]\n                if regime==\'shifted_validation\':va_hard=normalize(.8*va_hard+.2*np.roll(va_hard,1,axis=1))\n                score_pool=-np.log(va_pool.reshape(vb,4)[np.arange(vb),yval]).mean()\n                score_hard=-np.log(va_hard.reshape(vb,4)[np.arange(vb),yval]).mean()\n                chosen=\'hard\' if score_hard<score_pool else \'pool\'\n                ptest=(hard if chosen==\'hard\' else pooled)[-200:]\n                loss=-np.log(ptest.reshape(200,4)[np.arange(200),ytest]).mean()\n                loss_pool=-np.log(pooled[-200:].reshape(200,4)[np.arange(200),ytest]).mean()\n                rows.append({\'regime\':regime,\'repeat\':r,\'n_validation\':vb,\'chosen\':chosen,\n                             \'selected_minus_pool_nll\':float(loss-loss_pool),\'selected_nll\':float(loss),\'pool_nll\':float(loss_pool)})\n    new_df=pd.DataFrame(rows);dest=root/\'reports\'/\'SYNTHETIC_only_stress.csv\'\n    if dest.exists():\n        prior=pd.read_csv(dest)\n        new_df=pd.concat([prior,new_df],ignore_index=True).drop_duplicates([\'regime\',\'repeat\',\'n_validation\'],keep=\'last\')\n    new_df.sort_values([\'regime\',\'repeat\',\'n_validation\']).to_csv(dest,index=False)\n    return {\'status\':\'SYNTHETIC_COMPLETE\',\'new_rows\':len(rows),\'cumulative_rows\':len(new_df),\'path\':str(dest),\'not_real_model_evidence\':True}\n\ndef final_archive(root=None,strict=True):\n    root=init_dirs(root);cache_paths=sorted((root/\'real_caches\').glob(\'*.npz\'))\n    valid=[];invalid=[];datasets=set();models=set()\n    for p in cache_paths:\n        try:\n            _,meta=read_cache(p)\n            if meta[\'artifact_kind\']!=\'real_tfm\':raise ValueError(\'not a real TFM cache\')\n            valid.append(str(p));datasets.add(meta[\'dataset\']);models.add(meta[\'model\'])\n        except Exception as e:invalid.append({\'file\':str(p),\'reason\':str(e)})\n    reports=[\'analysis_status.json\',\'fixed_test_factorial.csv\',\'marginal_conditional_scores.csv\',\'dataset_blocks.csv\',\'dataset_blocked_uncertainty.csv\']\n    report_ok=all((root/\'reports\'/s).exists() for s in reports)\n    # Archive readiness deliberately stricter than successful notebook execution.\n    complete=bool(len(valid)>=8 and len(datasets)>=3 and {\'tabiclv2\',\'tabpfn3\'}<=models and report_ok and not invalid)\n    status={\'protocol\':PROTOCOL,\'status\':\'COMPLETE_PILOT_GATE\' if complete else \'PARTIAL_EVIDENCE_ONLY\',\n            \'valid_cache_count\':len(valid),\'datasets\':sorted(datasets),\'models\':sorted(models),\n            \'invalid\':invalid,\'report_files_complete\':report_ok,\n            \'complete_pilot_gate\':complete,\'full_paper_submission_ready\':False,\n            \'checkpoint_bytes_authenticated\':False,\n            \'notes\':\'Passing this modest pilot gate is not a full manuscript replication. Verify checkpoint weights and 10-dataset five-fold statistics separately.\'}\n    import zipfile\n    zip_path=root/\'reports\'/(\'CoReTFM_V2_PILOT_COMPLETE.zip\' if complete else \'CoReTFM_V2_PARTIAL_EVIDENCE.zip\')\n    status[\'zip_path\']=str(zip_path)\n    put_json(root/\'reports\'/\'EVIDENCE_GATE.json\',status)\n    # Archive outputs but exclude dataset pickle caches, secrets, model weights and temp files.\n    with zipfile.ZipFile(zip_path,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:\n        items=[]\n        for folder in [\'splits\',\'real_caches\',\'smoke_caches\',\'reports\',\'runtime\',\'failures\']:\n            for p in sorted((root/folder).rglob(\'*\')):\n                if p.is_file() and p.resolve()!=zip_path.resolve() and p.suffix.lower() not in (\'.ckpt\',\'.safetensors\',\'.token\',\'.zip\'):\n                    items.append(p)\n        for p in items:z.write(p,p.relative_to(root).as_posix())\n        manifest=[{\'path\':str(p.relative_to(root)),\'sha256\':sha_bytes(p.read_bytes()),\'bytes\':p.stat().st_size} for p in items]\n        z.writestr(\'ARCHIVE_SHA256_MANIFEST.json\',json.dumps(manifest,indent=2))\n    return status\n'
q=types.ModuleType('coretfm_embedded_v2')
exec(compile(ENGINE_SOURCE,'coretfm_engine.py','exec'),q.__dict__)
ROOT=q.init_dirs()
print('Output root:',ROOT,'Cache protocol:',q.PROTOCOL)


In [ ]:
# 4 — Research v4 extension. All selection is based on validation labels.
RESEARCH_SOURCE = '"""CoRe-TFM v4 research extension. Uses genuine q.read_cache four-view archives.\nAll model policy choices are validation-only; test labels are never used for selection.\nThe baseline engine protocol remains v2 for backwards cache compatibility.\n"""\nfrom __future__ import annotations\nimport hashlib, json, math, os, time, traceback, zipfile\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import wilcoxon, spearmanr\n\nMETHODS = ("arithmetic", "geometric", "raw_BA", "raw_AB", "hard", "soft_25", "soft_50", "soft_75")\nWEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)\nTEMPERATURES = (0.65, 0.8, 1.0, 1.25, 1.5, 2.0, 3.0)\nPOLICIES = ("fixed_arithmetic", "fixed_geometric", "raw_BA", "raw_AB", "hard", "soft_50",\n            "pool_only", "raw_or_pool", "repair_or_pool", "full_family", "guarded_full",\n            "temperature_pool", "weighted_pool", "weighted_temperature_pool")\n\n\ndef _write_json(p, obj):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_name(p.name+\'.writing\')\n    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str),encoding=\'utf-8\');tmp.replace(p)\n\ndef recover(zipfile_path, root, q):\n    """Restore authentic split/cache pairs only, never overwrite differing bytes.\n    Validate member paths, old manifest, and recovered cache; reports are recomputed.\n    """\n    root=Path(root); report=[]; allowed={\'splits\',\'real_caches\'}\n    zpath=Path(zipfile_path)\n    if not zpath.is_file():return {\'status\':\'ZIP_MISSING\',\'zip\':str(zpath)}\n    with zipfile.ZipFile(zpath) as z:\n        bad=z.testzip()\n        if bad:raise ValueError(\'Corrupt zip member \'+bad)\n        names=set(z.namelist())\n        if \'ARCHIVE_SHA256_MANIFEST.json\' in names:\n            manifest=json.loads(z.read(\'ARCHIVE_SHA256_MANIFEST.json\'))\n            for entry in manifest:\n                if entry[\'path\'] not in names:raise ValueError(\'Missing signed manifest member\')\n                data=z.read(entry[\'path\'])\n                if hashlib.sha256(data).hexdigest()!=entry[\'sha256\'] or len(data)!=entry[\'bytes\']:\n                    raise ValueError(\'Original archive manifest hash/size failed for \'+entry[\'path\'])\n        for entry in z.infolist():\n            if entry.is_dir():continue\n            rel=Path(entry.filename)\n            if (rel.is_absolute() or \'..\' in rel.parts or not rel.parts or\n                rel.parts[0] not in allowed or rel.suffix not in (\'.npz\',\'.json\')):continue\n            target=(root/rel).resolve()\n            if not target.is_relative_to(root.resolve()):raise ValueError(\'Unsafe path\')\n            new=z.read(entry)\n            target.parent.mkdir(parents=True,exist_ok=True)\n            if target.exists():\n                status=\'ALREADY_IDENTICAL\' if hashlib.sha256(target.read_bytes()).digest()==hashlib.sha256(new).digest() else \'CONFLICT_KEPT_EXISTING\'\n            else:\n                q._atomic_bytes(target,new);status=\'RESTORED\'\n            report.append({\'file\':rel.as_posix(),\'status\':status})\n    invalid=[]; valid=[]\n    for p in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            _,m=q.read_cache(p)\n            if m.get(\'artifact_kind\')!=\'real_tfm\':raise ValueError(\'Not real TFM\')\n            valid.append(str(p))\n        except Exception as e: invalid.append({\'file\':str(p),\'error\':str(e)})\n    for p in sorted((root/\'splits\').glob(\'*.json\')):\n        try:\n            s=q.get_json(p);q.assert_no_overlap(s)\n            if q.sha_obj({k:v for k,v in s.items() if k!=\'split_sha256\'})!=s.get(\'split_sha256\'):\n                raise ValueError(\'Split signature mismatch\')\n        except Exception as e:invalid.append({\'file\':str(p),\'error\':str(e)})\n    result={\'status\':\'RESTORED_AND_CHECKED\',\'actions\':report,\'valid_real_caches\':len(valid),\'invalid\':invalid}\n    _write_json(root/\'reports\'/\'V4_RECOVERY.json\',result)\n    return result\n\n\ndef plan_jobs(datasets,models,seeds,folds,caps):\n    """Coverage-first schedule: new datasets before extra caps, then independent splits.\n    Priority cap=128 first (64 on low label budgets), then 256, 64, 512, 1024.\n    """\n    ds=list(dict.fromkeys(datasets));mod=list(dict.fromkeys(models))\n    return [(d,m,int(s),int(f),int(c))\n            for s in seeds for f in folds\n            for c in sorted(caps,key=lambda n:(0 if int(n)==128 else 1 if int(n)==256 else 2, int(n)))\n            for d in ds for m in mod]\n\ndef _cache_path(root,job):\n    ds,m,s,f,c=job\n    return Path(root)/\'real_caches\'/f\'{ds}__{m}__seed{s}__fold{f}__train{c}.npz\'\n\ndef job_coverage(q,root,jobs):\n    rows=[]\n    for job in jobs:\n        path=_cache_path(root,job);status=\'MISSING\'; reason=\'\'\n        if path.exists():\n            try:\n                _,meta=q.read_cache(path)\n                if meta[\'artifact_kind\']!=\'real_tfm\' or (meta[\'dataset\'],meta[\'model\'],int(meta[\'seed\']),int(meta[\'fold\']),int(meta[\'train_cap\']))!=job:\n                    raise ValueError(\'Cache metadata identity mismatch\')\n                status=\'VERIFIED_REAL\'\n            except Exception as exc:status=\'INVALID\';reason=str(exc)\n        rows.append(dict(zip([\'dataset\',\'model\',\'seed\',\'fold\',\'train_cap\'],job)) | {\'status\':status,\'problem\':reason})\n    df=pd.DataFrame(rows)\n    df.to_csv(Path(root)/\'reports\'/\'V4_JOB_COVERAGE.csv\',index=False)\n    return df\n\n\ndef run_queue(q,root,jobs,ensure_package,token_present=False,max_new=1):\n    """Only one or a bounded number of NEW GPU runs. Auth errors are blockers, not successes."""\n    root=Path(root);coverage=job_coverage(q,root,jobs)\n    if max_new<1:raise ValueError(\'max_new must be positive\')\n    if not q.cuda_status().get(\'cuda\'):\n        status={\'status\':\'SKIPPED_NO_CUDA\',\'remaining\':int((coverage.status!=\'VERIFIED_REAL\').sum())}\n        _write_json(root/\'reports\'/\'V4_QUEUE_LAST.json\',status);return status\n    events=[];count=0\n    for job in jobs:\n        if count>=max_new:break\n        path=_cache_path(root,job)\n        if path.exists():\n            try:q.read_cache(path);continue\n            except Exception as e:\n                events.append({\'job\':job,\'status\':\'BLOCKED_CORRUPT_EXISTING_CACHE\',\'message\':str(e)});continue\n        ds,model,seed,fold,cap=job\n        if model==\'tabpfn3\' and not token_present:\n            events.append({\'job\':job,\'status\':\'BLOCKED_TABPFN_AUTH\',\'message\':\'Accept Prior Labs license and set TABPFN_TOKEN in Colab Secrets\'});continue\n        try:\n            pkg=ensure_package(model)\n            if pkg.get(\'status\')!=\'PACKAGE_READY\':events.append({\'job\':job,**pkg});continue\n            result=q.run_case(ds,model,seed,fold,cap,root=root,test_n=128,val_n=205)\n            events.append({\'job\':job,**result})\n            if result.get(\'status\')==\'REAL_MODEL_CACHE_WRITTEN\':count+=1\n        except Exception as exc:\n            # Do not include tokens or environment contents in the persisted traceback.\n            events.append({\'job\':job,\'status\':\'FAILED_INFERENCE\',\'exception_type\':type(exc).__name__,\n                           \'message\':str(exc)[:450]})\n            # A failed job should not prevent valid runs for other datasets/models.\n    result={\'status\':\'QUEUE_FINISHED\',\'new_real_caches\':count,\'attempted_or_blocked\':len(events),\'events\':events,\n            \'note\':\'Re-execute this cell to resume; no test label is used for selection.\'}\n    _write_json(root/\'reports\'/\'V4_QUEUE_LAST.json\',result)\n    job_coverage(q,root,jobs)\n    return result\n\n\ndef _normalize_joint(x):\n    x=np.maximum(np.asarray(x,dtype=np.float64),1e-12)\n    return x/x.sum(axis=(1,2),keepdims=True)\n\ndef _temperature(x,t):\n    x=np.maximum(np.asarray(x,dtype=np.float64),1e-12)\n    z=np.log(x)/float(t);z-=z.max(axis=(1,2),keepdims=True)\n    return _normalize_joint(np.exp(z))\n\ndef _means(q,p,ya,yb):\n    n,b=q.per_sample_metrics(p,ya,yb)\n    return float(n.mean()),float(b.mean())\n\ndef _ece_top(p,ya,yb,bins=10):\n    flat=p.reshape(len(p),-1)\n    conf=flat.max(axis=1);pred=flat.argmax(axis=1)\n    truth=np.asarray(ya)*p.shape[2]+np.asarray(yb)\n    partitions=np.minimum(np.floor(conf*bins).astype(int),bins-1)\n    return float(sum(((partitions==j).sum()/len(p))*abs((pred[partitions==j]==truth[partitions==j]).mean()-conf[partitions==j].mean())\n                     for j in range(bins) if (partitions==j).any()))\n\ndef evaluate_cache(q,path,budgets,penalty=0.15):\n    """Validation-only candidate/policy selection; test labels accessed only AFTER policies decided."""\n    d,m=q.read_cache(path)\n    if m.get(\'artifact_kind\')!=\'real_tfm\':raise ValueError(\'Not real-model evidence\')\n    val=q.candidates(d,\'val\');test=q.candidates(d,\'test\')\n    ya_v,yb_v=d[\'ya_val\'],d[\'yb_val\']; ya_t,yb_t=d[\'ya_test\'],d[\'yb_test\']\n    assert set(val)==set(test)==set(METHODS)\n    j1,j2=q.implied(d,\'test\')\n    tv=float(np.mean(.5*np.abs(j1-j2).sum(axis=(1,2))))\n    marg=float(.5*(np.mean(.5*np.abs(test[\'arithmetic\'].sum(axis=2)-d[\'a_test\']).sum(axis=1))+\n                   np.mean(.5*np.abs(test[\'arithmetic\'].sum(axis=1)-d[\'b_test\']).sum(axis=1))))\n    base_nll,base_brier=_means(q,test[\'arithmetic\'],ya_t,yb_t)\n    rows=[]\n    for requested_budget in budgets:\n        budget=int(requested_budget)\n        if budget>len(ya_v) or budget<8:continue\n        val_scores={k:_means(q,v[:budget],ya_v[:budget],yb_v[:budget])[0] for k,v in val.items()}\n        subsets={\n            \'pool_only\':[\'arithmetic\',\'geometric\'],\n            \'raw_or_pool\':[\'arithmetic\',\'geometric\',\'raw_BA\',\'raw_AB\'],\n            \'repair_or_pool\':[\'arithmetic\',\'geometric\',\'hard\',\'soft_25\',\'soft_50\',\'soft_75\'],\n            \'full_family\':list(METHODS),\n        }\n        choices={k:(k,{}) for k in (\'fixed_arithmetic\',\'fixed_geometric\',\'raw_BA\',\'raw_AB\',\'hard\',\'soft_50\')}\n        choices={p:((\'arithmetic\' if p==\'fixed_arithmetic\' else \'geometric\' if p==\'fixed_geometric\' else p),{}) for p in choices}\n        for policy,names in subsets.items():choices[policy]=(min(names,key=lambda k:val_scores[k]),{})\n        best=choices[\'full_family\'][0]\n        guard=penalty*math.sqrt(math.log(len(METHODS))/budget)\n        choices[\'guarded_full\']=(best if val_scores[\'arithmetic\']-val_scores[best]>guard else \'arithmetic\',{\'penalty\':guard})\n        candidates={}\n        # Temperature scaling: train only on same held-out validation pool, never fit to test.\n        for t in TEMPERATURES:\n            candidates[(\'temp\',t)]=_temperature(val[\'arithmetic\'][:budget],t)\n        temp=min(TEMPERATURES,key=lambda t:_means(q,candidates[(\'temp\',t)],ya_v[:budget],yb_v[:budget])[0])\n        choices[\'temperature_pool\']=(\'T\',{\'temperature\':temp})\n        best_weight=min(WEIGHTS,key=lambda w:_means(q,_normalize_joint(w*val[\'raw_BA\'][:budget]+(1-w)*val[\'raw_AB\'][:budget]),ya_v[:budget],yb_v[:budget])[0])\n        choices[\'weighted_pool\']=(\'W\',{\'weight_BA\':best_weight})\n        bestwt=min(((w,t) for w in WEIGHTS for t in TEMPERATURES),\n                   key=lambda wt:_means(q,_temperature(_normalize_joint(wt[0]*val[\'raw_BA\'][:budget]+(1-wt[0])*val[\'raw_AB\'][:budget]),wt[1]),ya_v[:budget],yb_v[:budget])[0])\n        choices[\'weighted_temperature_pool\']=(\'WT\',{\'weight_BA\':bestwt[0],\'temperature\':bestwt[1]})\n        for policy,(pick,hyp) in choices.items():\n            if pick==\'T\':pred=_temperature(test[\'arithmetic\'],hyp[\'temperature\'])\n            elif pick==\'W\':pred=_normalize_joint(hyp[\'weight_BA\']*test[\'raw_BA\']+(1-hyp[\'weight_BA\'])*test[\'raw_AB\'])\n            elif pick==\'WT\':pred=_temperature(_normalize_joint(hyp[\'weight_BA\']*test[\'raw_BA\']+(1-hyp[\'weight_BA\'])*test[\'raw_AB\']),hyp[\'temperature\'])\n            else:pred=test[pick]\n            nll,brier=_means(q,pred,ya_t,yb_t)\n            row={k:m[k] for k in (\'dataset\',\'model\',\'seed\',\'fold\',\'train_cap\',\'split_sha256\',\'test_ids_sha\',\'package_version\',\'checkpoint_identifier\',\'checkpoint_sha256\')}\n            row.update({\'val_budget\':budget,\'policy\':policy,\'selected\':pick,\'selected_params\':json.dumps(hyp),\n                        \'test_nll\':nll,\'test_brier\':brier,\'delta_nll\':nll-base_nll,\n                        \'delta_brier\':brier-base_brier,\'base_nll\':base_nll,\'base_brier\':base_brier,\n                        \'top1_ece_descriptive\':_ece_top(pred,ya_t,yb_t),\n                        \'top1_accuracy\':float((pred.reshape(len(pred),-1).argmax(axis=1)==ya_t*pred.shape[2]+yb_t).mean()),\n                        \'factor_tv\':tv,\'arithmetic_marginal_tv\':marg,\'n_test\':len(ya_t),\n                        \'n_val_available\':len(ya_v),\'cache_sha256\':m[\'npz_sha256\']})\n            rows.append(row)\n    return rows\n\n\ndef _paired_dataset_blocks(df):\n    # Strict paired-coverage gate: comparisons within a model/training/budget must use identical split cells.\n    data=[];gcols=[\'model\',\'train_cap\',\'val_budget\']\n    for key,g in df.groupby(gcols):\n        arithmetic=g[g.policy==\'fixed_arithmetic\'][[\'dataset\',\'seed\',\'fold\',\'test_ids_sha\',\'test_nll\']].rename(columns={\'test_nll\':\'baseline\'})\n        for policy,p in g.groupby(\'policy\'):\n            paired=p.merge(arithmetic,on=[\'dataset\',\'seed\',\'fold\',\'test_ids_sha\'],how=\'inner\',validate=\'one_to_one\')\n            if len(paired)!=len(p) or len(paired)!=len(arithmetic):\n                raise ValueError(\'Unmatched split coverage detected; will not silently aggregate\')\n            for ds,dsub in paired.groupby(\'dataset\'):\n                data.append(dict(zip(gcols,key))|{\'policy\':policy,\'dataset\':ds,\'delta_nll\':float((dsub.test_nll-dsub.baseline).mean()),\'n_splits\':len(dsub)})\n    return pd.DataFrame(data)\n\ndef analyze(q,root,budgets=(16,32,52,103,205),primary_budget=52,primary_cap=256):\n    root=Path(root); root.joinpath(\'reports\').mkdir(parents=True,exist_ok=True)\n    all_rows=[];errors=[];validmeta=[]\n    for p in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            _,m=q.read_cache(p)\n            if m[\'artifact_kind\']!=\'real_tfm\':raise ValueError(\'Smoke cache excluded\')\n            validmeta.append(m)\n            all_rows.extend(evaluate_cache(q,p,budgets))\n        except Exception as e:errors.append({\'file\':p.name,\'error\':str(e)})\n    if not all_rows:\n        report={\'status\':\'NOT_RUN_NO_VALID_REAL_EVIDENCE\',\'errors\':errors}\n        _write_json(root/\'reports\'/\'V4_ANALYSIS_STATUS.json\',report);return report\n    df=pd.DataFrame(all_rows)\n    # Never silently pool results from different releases or checkpoint identities.\n    mixed_package=df.groupby(\'model\').package_version.nunique()\n    mixed_checkpoint=df.groupby(\'model\').checkpoint_identifier.nunique()\n    if (mixed_package>1).any() or (mixed_checkpoint>1).any():\n        raise RuntimeError(\'Mixed model package/checkpoint versions. Separate the result protocols before inference.\')\n    # Holdouts must be frozen across model and training context within a named split.\n    if (df.groupby([\'dataset\',\'seed\',\'fold\'])[\'test_ids_sha\'].nunique()>1).any():\n        raise AssertionError(\'Mismatched test-set identities across training caps/models\')\n    df.to_csv(root/\'reports\'/\'V4_PER_CACHE_POLICY_RESULTS.csv\',index=False)\n    blocks=_paired_dataset_blocks(df)\n    blocks.to_csv(root/\'reports\'/\'V4_DATASET_BLOCKS.csv\',index=False)\n    summaries=[]\n    for (model,cap,budget,policy),g in blocks.groupby([\'model\',\'train_cap\',\'val_budget\',\'policy\']):\n        vals=g.delta_nll.to_numpy();n=len(vals)\n        lo=hi=pvalue=np.nan\n        if n>=3:\n            rng=np.random.default_rng(20261019)\n            boot=np.mean(rng.choice(vals,size=(4000,n),replace=True),axis=1)\n            lo,hi=np.quantile(boot,[.025,.975])\n        if n>=5 and np.any(vals!=0):\n            try:pvalue=float(wilcoxon(vals).pvalue)\n            except ValueError:pass\n        summaries.append({\'model\':model,\'train_cap\':cap,\'val_budget\':budget,\'policy\':policy,\n                          \'datasets\':n,\'delta_nll_mean\':float(vals.mean()),\'ci95_low\':lo,\'ci95_high\':hi,\n                          \'wilcoxon_p_unadjusted_exploratory\':pvalue,\'wins\':int((vals<0).sum()),\n                          \'dataset_unit_not_folds\':True})\n    res=pd.DataFrame(summaries)\n    # Family-wise Holm adjustment on each model/cap/budget among all NON-baseline policies.\n    res[\'holm_p_exploratory\']=np.nan\n    for key,ix in res.groupby([\'model\',\'train_cap\',\'val_budget\']).groups.items():\n        index=[j for j in ix if res.loc[j,\'policy\']!=\'fixed_arithmetic\' and np.isfinite(res.loc[j,\'wilcoxon_p_unadjusted_exploratory\'])]\n        pvals=sorted(index,key=lambda j:res.loc[j,\'wilcoxon_p_unadjusted_exploratory\'])\n        running=0.0;m=len(pvals)\n        for jrank,j in enumerate(pvals):\n            running=max(running,min(1.0,(m-jrank)*float(res.loc[j,\'wilcoxon_p_unadjusted_exploratory\'])))\n            res.loc[j,\'holm_p_exploratory\']=running\n    res.to_csv(root/\'reports\'/\'V4_DATASET_BLOCKED_INFERENCE.csv\',index=False)\n    # Reporting completeness: independent dataset/model coverage for primary comparison.\n    primary=res[(res.train_cap==primary_cap)&(res.val_budget==primary_budget)&(res.policy.isin([\'full_family\',\'temperature_pool\',\'weighted_pool\',\'weighted_temperature_pool\',\'pool_only\']))]\n    primary.to_csv(root/\'reports\'/\'V4_PRIMARY_COMPARISON.csv\',index=False)\n    provenance=[]\n    for m in validmeta:\n        provenance.append({k:m.get(k) for k in [\'dataset\',\'model\',\'seed\',\'fold\',\'train_cap\',\'package_version\',\'checkpoint_identifier\',\n                                               \'checkpoint_sha256\',\'checkpoint_revision_confirmed\',\'npz_sha256\',\'split_sha256\',\'device\']})\n    _write_json(root/\'reports\'/\'V4_PROVENANCE.json\',provenance)\n    status={\'status\':\'EXPLORATORY_REAL_MODEL_ANALYSIS\',\'real_cache_count\':len(validmeta),\'datasets\':sorted(df.dataset.unique().tolist()),\n            \'models\':sorted(df.model.unique().tolist()),\'invalid_caches\':errors,\n            \'primary_cap\':primary_cap,\'primary_validation_budget\':primary_budget,\n            \'primary_comparison_dataset_counts\':{str(r[\'model\'])+\'_\'+str(r[\'policy\']):int(r[\'datasets\']) for _,r in primary.iterrows()},\n            \'checkpoint_bytes_verified\':bool(validmeta and all(m.get(\'checkpoint_revision_confirmed\') and m.get(\'checkpoint_sha256\')!=\'NOT_VERIFIED\' for m in validmeta)),\n            \'paper_ready\':False,\'note\':\'Dataset-block inference exploratory; no post-hoc best-policy p-value is confirmatory.\'}\n    _write_json(root/\'reports\'/\'V4_ANALYSIS_STATUS.json\',status)\n    return status\n\n\ndef plot_reports(root):\n    root=Path(root)\n    file=root/\'reports\'/\'V4_PER_CACHE_POLICY_RESULTS.csv\'\n    if not file.exists():return \'NO_REAL_RESULTS\'\n    import matplotlib.pyplot as plt\n    df=pd.read_csv(file)\n    limited=df[(df.val_budget==52)&(df.policy.isin([\'fixed_arithmetic\',\'full_family\',\'temperature_pool\',\'weighted_pool\',\'weighted_temperature_pool\']))]\n    if limited.empty:limited=df[(df.val_budget==df.val_budget.min())&(df.policy.isin([\'fixed_arithmetic\',\'full_family\',\'temperature_pool\']))]\n    fig,ax=plt.subplots(figsize=(9,4.8))\n    for (dataset,model),g in limited.groupby([\'dataset\',\'model\']):\n        base=g[g.policy==\'fixed_arithmetic\'].groupby(\'train_cap\').test_nll.mean().sort_index()\n        if not base.empty:ax.plot(base.index,base.values,marker=\'o\',label=f\'{dataset} / {model}\')\n    ax.set(xlabel=\'Training-context cap\',ylabel=\'Arithmetic pool joint NLL\',title=\'Real four-view model evidence: fixed-test learning curves\')\n    ax.grid(alpha=.25);ax.legend(fontsize=7,loc=\'best\');fig.tight_layout()\n    fp=root/\'reports\'/\'V4_REAL_CONTEXT_NLL.png\';fig.savefig(fp,dpi=180);plt.close(fig)\n    return str(fp)\n\n\ndef final_archive(root,q,planned_jobs=None):\n    root=Path(root);root.joinpath(\'reports\').mkdir(parents=True,exist_ok=True)\n    caches=[];invalid=[]\n    for p in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            _,m=q.read_cache(p)\n            if m.get(\'artifact_kind\')!=\'real_tfm\':raise ValueError(\'not a TFM\')\n            caches.append(m)\n        except Exception as e:invalid.append({\'path\':p.name,\'error\':str(e)})\n    datasets={x[\'dataset\'] for x in caches}; models={x[\'model\'] for x in caches}\n    gate={\'status\':\'PARTIAL_EVIDENCE_ONLY\',\'verified_real_caches\':len(caches),\'datasets\':sorted(datasets),\'models\':sorted(models),\n          \'invalid_cache_files\':invalid,\'checkpoint_hashes_verified\':bool(caches and all(m.get(\'checkpoint_revision_confirmed\') for m in caches)),\n          \'paper_submission_ready\':False,\'planned_jobs\':len(planned_jobs) if planned_jobs else None,\n          \'claim_limit\':\'Even sufficient cache counts cannot establish a publishable gain without dataset-level statistical analysis, matched coverage and provenance.\'}\n    _write_json(root/\'reports\'/\'V4_EVIDENCE_GATE.json\',gate)\n    files=[]\n    # Only the current V4 reports: stale V2/V3 summaries cannot masquerade as new evidence.\n    for folder in (\'splits\',\'real_caches\'):\n        for p in sorted((root/folder).rglob(\'*\')):\n            if p.is_file() and p.suffix.lower() in (\'.json\',\'.npz\') and not p.name.endswith(\'.writing\'):\n                files.append(p)\n    for p in sorted((root/\'reports\').glob(\'V4_*\')):\n        if p.is_file() and p.suffix.lower() in (\'.json\',\'.csv\',\'.png\',\'.txt\',\'.md\'):\n            files.append(p)\n    time_report=root/\'runtime\'/\'cache_runtime.csv\'\n    if time_report.is_file():files.append(time_report)\n    archive=root/\'reports\'/\'CoReTFM_V4_RESEARCH_EVIDENCE.zip\'\n    # No model weights, dataset table pickle, token strings, or archived duplicate ZIPs included.\n    manifest=[{\'path\':str(p.relative_to(root).as_posix()),\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest(),\n               \'bytes\':p.stat().st_size} for p in files]\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED,compresslevel=5) as z:\n        for p in files:z.write(p,p.relative_to(root).as_posix())\n        z.writestr(\'ARCHIVE_SHA256_MANIFEST.json\',json.dumps(manifest,indent=2))\n    with zipfile.ZipFile(archive) as z:\n        if z.testzip():raise RuntimeError(\'ZIP integrity failed\')\n        for row in manifest:\n            b=z.read(row[\'path\'])\n            if hashlib.sha256(b).hexdigest()!=row[\'sha256\'] or len(b)!=row[\'bytes\']:\n                raise RuntimeError(\'Archive content hash mismatch\')\n    gate[\'zip_path\']=str(archive);gate[\'files\']=len(files);gate[\'zip_size_mb\']=round(archive.stat().st_size/2**20,2)\n    return gate\n\n\ndef validation_selection_stability(q,root,budgets=(16,52,103,205),repeats=30):\n    """Repeated subsamples of existing real-model validation sets.\n    Not new independent real-model training runs; test outcomes are diagnostic only.\n    """\n    root=Path(root);rows=[];invalid=[]\n    for path in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            d,m=q.read_cache(path)\n            if m[\'artifact_kind\']!=\'real_tfm\':continue\n            V=q.candidates(d,\'val\');T=q.candidates(d,\'test\')\n            for budget in budgets:\n                if budget>len(d[\'ya_val\']) or budget<8:continue\n                for repeat in range(repeats):\n                    rng=np.random.default_rng(20261019+repeat*10007+budget*167+int(m[\'seed\'])*17+int(m[\'fold\']))\n                    selection=rng.choice(len(d[\'ya_val\']),size=budget,replace=False)\n                    scores={k:_means(q,x[selection],d[\'ya_val\'][selection],d[\'yb_val\'][selection])[0] for k,x in V.items()}\n                    subsets={\n                        \'pool_only\':[\'arithmetic\',\'geometric\'],\n                        \'raw_or_pool\':[\'arithmetic\',\'geometric\',\'raw_BA\',\'raw_AB\'],\n                        \'full_family\':list(METHODS),\n                    }\n                    for policy,allowed in subsets.items():\n                        best=min(allowed,key=scores.get)\n                        score=_means(q,T[best],d[\'ya_test\'],d[\'yb_test\'])[0]\n                        baseline=_means(q,T[\'arithmetic\'],d[\'ya_test\'],d[\'yb_test\'])[0]\n                        rows.append({\'dataset\':m[\'dataset\'],\'model\':m[\'model\'],\'seed\':m[\'seed\'],\'fold\':m[\'fold\'],\n                                     \'train_cap\':m[\'train_cap\'],\'validation_budget\':budget,\'subsample_repeat\':repeat,\n                                     \'policy\':policy,\'selected\':best,\'test_delta_nll\':score-baseline,\n                                     \'note\':\'real-model validation-resampling diagnostic; NOT a new independent TFM run\'})\n        except Exception as e:invalid.append({\'file\':path.name,\'error\':str(e)})\n    if rows:\n        pd.DataFrame(rows).to_csv(root/\'reports\'/\'V4_REAL_VALIDATION_SELECTION_STABILITY.csv\',index=False)\n    result={\'status\':\'REAL_CACHE_VALIDATION_RESAMPLING\' if rows else \'NO_REAL_CACHES\',\'rows\':len(rows),\n            \'real_model_inference_jobs_added\':0,\'invalid_caches\':invalid}\n    _write_json(root/\'reports\'/\'V4_VALIDATION_STABILITY_STATUS.json\',result)\n    return result\n\n\ndef view_reliability_scores(q,root):\n    """Real-model per-class and conditional-vs-direct proper scores, no model fitting."""\n    root=Path(root);rows=[]\n    for p in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            d,m=q.read_cache(p)\n            if m.get(\'artifact_kind\')!=\'real_tfm\':continue\n            for target in (\'a\',\'b\'):\n                if target==\'a\':\n                    labels=d[\'ya_test\'];direct=d[\'a_test\']; conditional=d[\'a_given_b_test\'][np.arange(len(labels)),d[\'yb_test\'],:]\n                else:\n                    labels=d[\'yb_test\'];direct=d[\'b_test\'];conditional=d[\'b_given_a_test\'][np.arange(len(labels)),d[\'ya_test\'],:]\n                for kind,pred in [(\'direct\',direct),(\'observed_conditioned\',conditional)]:\n                    for label in range(pred.shape[1]):\n                        mask=labels==label\n                        if not mask.any():continue\n                        nll=-np.log(np.maximum(pred[mask,label],1e-12))\n                        brier=(pred[mask]**2).sum(axis=1)+1-2*pred[mask,label]\n                        rows.append({\'dataset\':m[\'dataset\'],\'model\':m[\'model\'],\'seed\':m[\'seed\'],\'fold\':m[\'fold\'],\n                                     \'train_cap\':m[\'train_cap\'],\'target\':target.upper(),\'view\':kind,\'class_index\':label,\n                                     \'support_test\':int(mask.sum()),\'class_nll\':float(nll.mean()),\n                                     \'class_brier\':float(brier.mean()),\'rare_support_under_5\':bool(mask.sum()<5),\n                                     \'note\':\'Conditional proper score conditions on observed other target; diagnostic only.\'})\n        except Exception:continue\n    if rows:pd.DataFrame(rows).to_csv(root/\'reports\'/\'V4_CLASSWISE_VIEW_RELIABILITY.csv\',index=False)\n    return {\'status\':\'CLASSWISE_REAL\' if rows else \'NO_REAL_CACHE\',\'rows\':len(rows)}\n'
v4=types.ModuleType('coretfm_research_v4')
exec(compile(RESEARCH_SOURCE,'research_v4.py','exec'),v4.__dict__)
print('v4 code SHA256:',hashlib.sha256(RESEARCH_SOURCE.encode()).hexdigest())


## Recover, inspect and freeze prior evidence

In [ ]:
# 5 — Recover verified previous results (optional). No overwrite of differing cache bytes.
from pathlib import Path
existing=sorted(Path('/content').glob('CoReTFM_V2_PARTIAL_EVIDENCE*.zip')) if IN_COLAB else []
recover_path=RECOVERY_ZIP or (str(max(existing,key=lambda p:p.stat().st_mtime)) if existing else None)
if recover_path:
    try:print(json.dumps(v4.recover(recover_path,ROOT,q),indent=2)[:3000])
    except Exception as e:
        print('RECOVERY_FAILED:',repr(e));raise
else:print('NO_UPLOADED_RECOVERY_ZIP; existing Drive caches will be validated directly.')


In [ ]:
# 7 — Queue coverage first, without downloading anything or loading model checkpoints.
JOBS=v4.plan_jobs(DATASETS,MODELS,SEEDS,FOLDS,TRAIN_CAPS)
coverage=v4.job_coverage(q,ROOT,JOBS)
print('Jobs planned:',len(JOBS))
print(coverage.status.value_counts().to_string())
print('Next jobs:')
display(coverage[coverage.status!='VERIFIED_REAL'].head(12))


## Baselines, validity and inference

In [ ]:
# 9 — Independent analysis of authentic four-view caches, CPU okay.
# Strong baselines: fixed / validation-selected pools, raw, CoRe, weighted pools, temperature scaling.
if RUN_REAL_ANALYSIS:
    result=v4.analyze(q,ROOT,budgets=VAL_BUDGETS,primary_budget=PRIMARY_VAL,primary_cap=PRIMARY_CAP)
    print(json.dumps(result,indent=2)[:5000])
    path=ROOT/'reports'/'V4_PRIMARY_COMPARISON.csv'
    if path.exists():
        import pandas as pd
        display(pd.read_csv(path).head(30))
else:print('ANALYSIS_DISABLED')


In [ ]:
# 9B — Real-model validation-selection stability: finite-label sensitivity.
# Repeated validation subsamples are correlated diagnostics, NOT new model inference.
if RUN_REAL_ANALYSIS:
    print('Stability:',v4.validation_selection_stability(q,ROOT,repeats=30))
    print('Classwise:',v4.view_reliability_scores(q,ROOT))


## Stage dependencies
This notebook is standalone, but for fresh model results run the GPU queue notebook first (or use the all-in-one). Other stages read verified caches from the shared output directory.